<a href="https://colab.research.google.com/github/lipikamallikarjuna/MissingRfidDetection-ANSC6060/blob/main/DataAnlysis%26Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ydata-profiling

In [ ]:
from pathlib import Path
import pandas as pd
import pandas as pd
from ydata_profiling import ProfileReport
from pathlib import Path

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
import numpy as np

In [ ]:
project_root = Path("/content/drive/MyDrive/MiniProject")
data_file_path = project_root / "Data_set_prep_assignment_1.csv"
output_file_path = project_root / "MiniProjectMetaData.xlsx"

df = pd.read_csv(data_file_path, low_memory=False, dtype={"AnimalId": "string"})
df = df.rename(columns={
    "Avgmilkflow": "AverageMilkFlowKgPerMin",
    "Flow30_60Session": "MilkFlow30To60SecondsKgPerMin",
    "YieldFirst2Min_Session": "MilkYieldFirst2MinutesKg",
    "YieldSession": "TotalMilkYieldSessionKg",
    "DurationSession_sec": "MilkingDurationSeconds",
    "milking": "MilkingSession"
})
df["EventDate"] = pd.to_datetime(df["EventDate"], errors="coerce")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

Rows: 8,495,421
Columns: 11


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,AverageMilkFlowKgPerMin,MilkFlow30To60SecondsKgPerMin,MilkYieldFirst2MinutesKg,TotalMilkYieldSessionKg,MilkingDurationSeconds,MilkingSession
0,-8839528597343470980,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5365332022005618386,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,<NA>,NaN,NaN,NaN,2020-08-24,3.220506,3.501733,7.547777,14.560315,270,1
3,7750423760892252046,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
4,<NA>,NaN,NaN,NaN,2021-03-21,4.218409,5.098378,9.198853,13.970645,198,2


In [ ]:
cleaned_data_file_path = project_root / "cleaned_data.csv"
print(f"Cleaned data will be saved to: {cleaned_data_file_path}")

Cleaned data will be saved to: /content/drive/MyDrive/MiniProject/cleaned_data.csv


In [ ]:
definitions = {
    "AnimalId": "Animal identifier",
    "LactationNumber": "Lactation number",
    "DaysInMilk": "Days since calving",
    "ReproductionStatus": "Reproductive-status category",
    "EventDate": "Date of the milking event",
    "AverageMilkFlowKgPerMin": "Average milk flow rate",
    "MilkFlow30To60SecondsKgPerMin": "Flow rate from 30-60 seconds into milking",
    "MilkYieldFirst2MinutesKg": "Milk produced during the first 2 minutes",
    "TotalMilkYieldSessionKg": "Total milk produced during the session",
    "MilkingDurationSeconds": "Session duration in seconds",
    "MilkingSession": "Milking/session indicator or category"
}
canonical_units = {
    "AnimalId": "Identifier", "LactationNumber": "Count", "DaysInMilk": "Days",
    "ReproductionStatus": "Category", "EventDate": "Date",
    "AverageMilkFlowKgPerMin": "kg/min", "MilkFlow30To60SecondsKgPerMin": "kg/min",
    "MilkYieldFirst2MinutesKg": "kg", "TotalMilkYieldSessionKg": "kg",
    "MilkingDurationSeconds": "seconds", "MilkingSession": "Category"
}

metadata_rows = []
for column_name in df.columns:
    series = df[column_name]
    if pd.api.types.is_numeric_dtype(series):
        minimum, maximum = series.min(), series.max()
        value_range = maximum - minimum
    elif pd.api.types.is_datetime64_any_dtype(series):
        minimum = series.min().date().isoformat()
        maximum = series.max().date().isoformat()
        value_range = (series.max() - series.min()).days
    else:
        minimum = maximum = value_range = "N/A"

    metadata_rows.append({
        "Current name": column_name,
        "Suggested name": column_name,
        "Definition": definitions[column_name],
        "Total count": len(series),
        "Non-missing count": series.notna().sum(),
        "Missing count": series.isna().sum(),
        "Missing percent": round(series.isna().mean() * 100, 2),
        "Data type": str(series.dtype),
        "Unique values": series.nunique(dropna=True),
        "Minimum": minimum,
        "Maximum": maximum,
        "Range": value_range,
        "Canonical unit": canonical_units[column_name],
        "Example values": ", ".join(series.dropna().astype(str).unique()[:5])
    })

df_metadata_completed = pd.DataFrame(metadata_rows)
display(df_metadata_completed)

,Current name,Suggested name,Definition,Total count,Non-missing count,Missing count,Missing percent,Data type,Unique values,Minimum,Maximum,Range,Canonical unit,Example values
0,AnimalId,AnimalId,Animal identifier,8495421,6794418,1701003,20.02,string,9087,N/A,N/A,N/A,Identifier,"-8839528597343470980, -5365332022005618386, 77..."
1,LactationNumber,LactationNumber,Lactation number,8495421,6794418,1701003,20.02,float64,12,1.0,12.0,11.0,Count,"1.0, 5.0, 4.0, 2.0, 3.0"
2,DaysInMilk,DaysInMilk,Days since calving,8495421,6794414,1701007,20.02,float64,870,1.0,870.0,869.0,Days,"365.0, 66.0, 244.0, 394.0, 288.0"
3,ReproductionStatus,ReproductionStatus,Reproductive-status category,8495421,6794418,1701003,20.02,object,4,N/A,N/A,N/A,Category,"Pregnant, Bred, Open, Fresh"
4,EventDate,EventDate,Date of the milking event,8495421,8495421,0,0.00,datetime64[ns],830,2019-06-14,2021-10-24,863,Date,"2019-12-09, 2021-07-12, 2020-08-24, 2019-12-04..."
5,AverageMilkFlowKgPerMin,AverageMilkFlowKgPerMin,Average milk flow rate,8495421,8495249,172,0.00,float64,219,0.0,23.496085,23.496085,kg/min,"2.902991168, 3.311224301, 3.220505827, 3.90089..."
6,MilkFlow30To60SecondsKgPerMin,MilkFlow30To60SecondsKgPerMin,Flow rate from 30-60 seconds into milking,8495421,8495421,0,0.00,float64,102,0.0,10.500663,10.500663,kg/min,"0.8981128926, 1.4016004233, 3.5017330964, 4.10..."
7,MilkYieldFirst2MinutesKg,MilkYieldFirst2MinutesKg,Milk produced during the first 2 minutes,8495421,8495421,0,0.00,float64,1675,2.000342,11.997518,9.997176,kg,"4.9759082989, 5.3478540423, 7.5477770368, 8.40..."
8,TotalMilkYieldSessionKg,TotalMilkYieldSessionKg,Total milk produced during the session,8495421,8495421,0,0.00,float64,1010,5.034875,54.839318,49.804442,kg,"11.158372302000002, 15.059266684000002, 14.560..."
9,MilkingDurationSeconds,MilkingDurationSeconds,Session duration in seconds,8495421,8495421,0,0.00,int64,604,100,785,685,seconds,"226, 268, 270, 231, 198"


In [ ]:
df_metadata_completed.to_excel(output_file_path, sheet_name="Metadata", index=False)
print(f"Saved completed metadata to:\n{output_file_path}")

Saved completed metadata to:
/content/drive/MyDrive/MiniProject/MiniProjectMetaData.xlsx


In [ ]:

# Ensure project_root is defined from previous cells
# If running this cell independently, uncomment and set project_root
# project_root = Path("/content/drive/MyDrive/MiniProject")

# Generate the profile report
#print("Generating profile report")
#profile = ProfileReport(df, title="Dairy Farm Data Profile Report")

# Define the output path for the HTML report
#output_report_path = project_root / "DairyFarmData_ProfileReport.html"

# Save the report as an HTML file
#profile.to_file(output_report_path)

#print(f"Profile report saved to: {output_report_path}")

In [ ]:
# Data Quality Audit
numeric_columns = [
    "LactationNumber", "DaysInMilk",
    "AverageMilkFlowKgPerMin", "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg", "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds"
]

data_quality = pd.DataFrame({
    "DataType":        df.dtypes.astype(str),
    "MissingCount":    df.isna().sum(),
    "MissingPercent":  (df.isna().mean() * 100).round(2),
    "UniqueValues":    df.nunique(dropna=True),
    "NegativeCount":   df[numeric_columns].lt(0).sum().reindex(df.columns, fill_value=0),
    "ZeroCount":       df[numeric_columns].eq(0).sum().reindex(df.columns, fill_value=0),
})
data_quality.loc["EventDate", "InvalidDateCount"] = df["EventDate"].isna().sum()
data_quality.loc["MilkingSession", "OutsideRangeCount"] = (~df["MilkingSession"].isin([1,2,3])).sum()

print("Data Quality Audit")
display(data_quality.sort_values("MissingPercent", ascending=False))

print("\nDataset Summary")
display(pd.DataFrame({
    "Metric": ["Total records", "Total columns", "Date range start",
               "Date range end", "Unique AnimalIds"],
    "Value":  [len(df), df.shape[1], df["EventDate"].min().date(),
               df["EventDate"].max().date(), df["AnimalId"].nunique(dropna=True)]
}))

Data Quality Audit


,DataType,MissingCount,MissingPercent,UniqueValues,NegativeCount,ZeroCount,InvalidDateCount,OutsideRangeCount
AnimalId,string,1701003,20.02,9087,0,0,NaN,NaN
LactationNumber,float64,1701003,20.02,12,0,0,NaN,NaN
DaysInMilk,float64,1701007,20.02,870,0,0,NaN,NaN
ReproductionStatus,object,1701003,20.02,4,0,0,NaN,NaN
EventDate,datetime64[ns],0,0.00,830,0,0,0.0,NaN
AverageMilkFlowKgPerMin,float64,172,0.00,219,0,215,NaN,NaN
MilkFlow30To60SecondsKgPerMin,float64,0,0.00,102,0,3155,NaN,NaN
MilkYieldFirst2MinutesKg,float64,0,0.00,1675,0,0,NaN,NaN
TotalMilkYieldSessionKg,float64,0,0.00,1010,0,0,NaN,NaN
MilkingDurationSeconds,int64,0,0.00,604,0,0,NaN,NaN



Dataset Summary


,Metric,Value
0,Total records,8495421
1,Total columns,11
2,Date range start,2019-06-14
3,Date range end,2021-10-24
4,Unique AnimalIds,9087


In [ ]:
# Missingness Pattern
id_columns = ["AnimalId", "LactationNumber", "DaysInMilk", "ReproductionStatus"]

missing_pattern = (
    df[id_columns]
    .isna().astype(int)
    .value_counts()
    .reset_index(name="RecordCount")
)
print("Missingness Combinations (0=present, 1=missing)")
display(missing_pattern)

# Confirm: are all 4 always missing together or are there partial cases?
partial = df[id_columns].isna().sum(axis=1)
partial_counts = partial.value_counts().sort_index()
print("\nNumber of identifier columns missing per row:")
display(partial_counts.rename("RecordCount").to_frame())

n_patterns = missing_pattern.shape[0]
if n_patterns <= 2:
    print("\n✓ Clean block-missingness confirmed: all 4 identifiers missing together.")
else:
    print(f"\n⚠ {n_patterns} missingness patterns detected — partial missingness exists, investigate.")

Missingness Combinations (0=present, 1=missing)


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,RecordCount
0,0,0,0,0,6794414
1,1,1,1,1,1701003
2,0,0,1,0,4



Number of identifier columns missing per row:


,RecordCount
0,6794414
1,4
4,1701003



⚠ 3 missingness patterns detected — partial missingness exists, investigate.


In [ ]:
# Known vs Missing AnimalId
df_master = df.copy()
df_master["AnimalIdMissing"] = df_master["AnimalId"].isna()

comparison_columns = [
    "AverageMilkFlowKgPerMin", "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg", "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds", "MilkingSession"
]

known_stats   = df_master.loc[~df_master["AnimalIdMissing"], comparison_columns].describe().T
missing_stats = df_master.loc[ df_master["AnimalIdMissing"], comparison_columns].describe().T

comparison = known_stats[["count","mean","std","min","50%","max"]].join(
    missing_stats[["count","mean","std","min","50%","max"]],
    lsuffix="_KnownId", rsuffix="_MissingId"
)
print("Feature Distribution: Known vs Missing AnimalId")
display(comparison)

Feature Distribution: Known vs Missing AnimalId


,count_KnownId,mean_KnownId,std_KnownId,min_KnownId,50%_KnownId,max_KnownId,count_MissingId,mean_MissingId,std_MissingId,min_MissingId,50%_MissingId,max_MissingId
AverageMilkFlowKgPerMin,6794287.0,3.525899,0.729146,0.000000,3.492661,23.496085,1700962.0,3.526221,0.729128,0.000000,3.492661,23.496085
MilkFlow30To60SecondsKgPerMin,6794418.0,3.751396,1.485795,0.000000,3.900894,10.500663,1701003.0,3.752198,1.485216,0.000000,3.900894,10.101502
MilkYieldFirst2MinutesKg,6794418.0,7.490827,2.014306,2.000342,7.525097,11.997518,1701003.0,7.492219,2.013337,2.000342,7.525097,11.997518
TotalMilkYieldSessionKg,6794418.0,13.928592,3.645881,5.034875,13.743849,54.839318,1701003.0,13.927960,3.646393,5.034875,13.743849,53.977492
MilkingDurationSeconds,6794418.0,236.870908,54.844360,100.000000,231.000000,785.000000,1701003.0,236.835210,54.838978,100.000000,231.000000,773.000000
MilkingSession,6794418.0,2.000101,0.816531,1.000000,2.000000,3.000000,1701003.0,1.999598,0.816361,1.000000,2.000000,3.000000


In [ ]:
# Animal Record & Date Profile
animal_records = (
    df_master[df_master["AnimalId"].notna()]
    .groupby("AnimalId")
    .agg(
        FirstDate      = ("EventDate", "min"),
        LastDate       = ("EventDate", "max"),
        UniqueDates    = ("EventDate", "nunique"),
        TotalRecords   = ("EventDate", "size")
    )
)

print("Per-Animal Record Summary")
display(animal_records.describe())

thresholds = pd.DataFrame({
    "Threshold":    ["< 10 records", "10–100 records", "> 100 records", "> 500 records"],
    "AnimalCount":  [
        (animal_records["TotalRecords"] < 10).sum(),
        ((animal_records["TotalRecords"] >= 10) & (animal_records["TotalRecords"] <= 100)).sum(),
        (animal_records["TotalRecords"] > 100).sum(),
        (animal_records["TotalRecords"] > 500).sum(),
    ]
})
print("\nRecord-count thresholds")
display(thresholds)

print("\nSession Distribution")
display(df_master.groupby("MilkingSession").size().rename("RecordCount").to_frame())

Per-Animal Record Summary


,FirstDate,LastDate,UniqueDates,TotalRecords
count,9087,9087,9087.000000,9087.000000
mean,2020-01-14 06:52:58.078574080,2021-04-02 00:12:02.614724352,296.540993,747.707494
min,2019-06-14 00:00:00,2019-06-15 00:00:00,1.000000,1.000000
25%,2019-06-15 00:00:00,2020-09-11 00:00:00,123.000000,313.000000
50%,2019-07-25 00:00:00,2021-09-27 00:00:00,279.000000,704.000000
75%,2020-07-30 00:00:00,2021-10-24 00:00:00,471.000000,1186.000000
max,2021-10-24 00:00:00,2021-10-24 00:00:00,701.000000,1763.000000
std,NaN,NaN,195.889120,492.065506



Record-count thresholds


,Threshold,AnimalCount
0,< 10 records,123
1,10–100 records,918
2,> 100 records,8046
3,> 500 records,5775



Session Distribution


,RecordCount
MilkingSession,
1,2831807
2,2831807
3,2831807


In [ ]:
# Outlier Flagging
numeric_milking = [
    "AverageMilkFlowKgPerMin", "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg", "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds"
]

flag_rows = {}

for col in numeric_milking:
    Q1  = df_master[col].quantile(0.25)
    Q3  = df_master[col].quantile(0.75)
    IQR = Q3 - Q1
    low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    flag_rows[f"{col}_IQR_Low"]  = df_master[col] < low
    flag_rows[f"{col}_IQR_High"] = df_master[col] > high
    print(f"{col}: IQR fence [{low:.3f}, {high:.3f}]")

# Domain hard flags
flag_rows["DaysInMilkOver350"]      = df_master["DaysInMilk"] > 350
flag_rows["DurationUnder150s"]      = df_master["MilkingDurationSeconds"] < 150
flag_rows["DurationOver600s"]       = df_master["MilkingDurationSeconds"] > 600
flag_rows["AvgFlowIsZero"]          = df_master["AverageMilkFlowKgPerMin"] == 0
flag_rows["Flow30To60IsZero"]         = df_master["MilkFlow30To60SecondsKgPerMin"] == 0
flag_rows["LactationNumberInvalid"] = ~df_master["LactationNumber"].isin(range(1,13))

outlier_flags = pd.DataFrame(flag_rows, index=df_master.index)

print("\nOutlier / Flag Counts")
display(outlier_flags.sum().rename("FlaggedRows").to_frame())

AverageMilkFlowKgPerMin: IQR fence [1.497, 5.488]
MilkFlow30To60SecondsKgPerMin: IQR fence [-0.352, 8.049]
MilkYieldFirst2MinutesKg: IQR fence [1.560, 13.463]
TotalMilkYieldSessionKg: IQR fence [4.082, 23.678]
MilkingDurationSeconds: IQR fence [84.500, 384.500]

Outlier / Flag Counts


,FlaggedRows
AverageMilkFlowKgPerMin_IQR_Low,3474
AverageMilkFlowKgPerMin_IQR_High,3411
MilkFlow30To60SecondsKgPerMin_IQR_Low,0
MilkFlow30To60SecondsKgPerMin_IQR_High,1560
MilkYieldFirst2MinutesKg_IQR_Low,0
MilkYieldFirst2MinutesKg_IQR_High,0
TotalMilkYieldSessionKg_IQR_Low,0
TotalMilkYieldSessionKg_IQR_High,43802
MilkingDurationSeconds_IQR_Low,0
MilkingDurationSeconds_IQR_High,50700


In [ ]:
# Metadata Rulebook
rulebook = {
    "AnimalId":                      (None, None,  "none",   False, "BlockWithAnimalId", True,  "Hashed int IDs; negatives valid; TARGET column"),
    "LactationNumber":               (1,    12,    "domain", False, "BlockWithAnimalId", True,  ">12 biologically impossible"),
    "DaysInMilk":                    (1,    870,   "IQR",    False, "BlockWithAnimalId", True,  ">350 soft flag — past typical dry-off"),
    "ReproductionStatus":            (None, None,  "none",   False, "BlockWithAnimalId", True,  "4 valid categories: Pregnant, Bred, Fresh, Open"),
    "EventDate":                     (None, None,  "domain", False, "None",              True,  "No missing; 2019-06-14 to 2021-10-24"),
    "AverageMilkFlowKgPerMin":       (0,    23.5,  "IQR",    False, "Independent (172)", True,  "Zero = likely sensor error"),
    "MilkFlow30To60SecondsKgPerMin": (0,    10.5,  "IQR",    True,  "None",              True,  "3155 zeros — investigate slow let-down vs sensor"),
    "MilkYieldFirst2MinutesKg":      (2.0,  12.0,  "IQR",    False, "None",              True,  "Min 2.0003 suggests sensor floor"),
    "TotalMilkYieldSessionKg":       (5.0,  54.8,  "IQR",    False, "None",              True,  "High corr with Duration; max 54.8 extreme"),
    "MilkingDurationSeconds":        (100,  785,   "IQR",    False, "None",              True,  "<150s and >600s are soft flags"),
    "MilkingSession":                (1,    3,     "domain", False, "None",              True,  "Uniform across 1/2/3; only 3 valid values"),
}

rulebook_rows = []
for col, (vmin, vmax, method, zero_ok, miss_pat, use_model, notes) in rulebook.items():
    base = df_metadata_completed[df_metadata_completed["Current name"] == col].iloc[0].to_dict()
    base.update({
        "ValidMin":         vmin,
        "ValidMax":         vmax,
        "OutlierMethod":    method,
        "ZeroAllowed":      zero_ok,
        "MissingPattern":   miss_pat,
        "UseInModel":       use_model,
        "Notes":            notes,
    })
    rulebook_rows.append(base)

df_rulebook = pd.DataFrame(rulebook_rows)

rulebook_path = project_root / "MiniProjectRulebook.xlsx"
df_rulebook.to_excel(rulebook_path, sheet_name="Rulebook", index=False)

print(f"Rulebook saved to: {rulebook_path}")
display(df_rulebook[["Current name","ValidMin","ValidMax","OutlierMethod",
                      "ZeroAllowed","MissingPattern","UseInModel","Notes"]])


Rulebook saved to: /content/drive/MyDrive/MiniProject/MiniProjectRulebook.xlsx


,Current name,ValidMin,ValidMax,OutlierMethod,ZeroAllowed,MissingPattern,UseInModel,Notes
0,AnimalId,NaN,NaN,none,False,BlockWithAnimalId,True,Hashed int IDs; negatives valid; TARGET column
1,LactationNumber,1.0,12.0,domain,False,BlockWithAnimalId,True,>12 biologically impossible
2,DaysInMilk,1.0,870.0,IQR,False,BlockWithAnimalId,True,>350 soft flag — past typical dry-off
3,ReproductionStatus,NaN,NaN,none,False,BlockWithAnimalId,True,"4 valid categories: Pregnant, Bred, Fresh, Open"
4,EventDate,NaN,NaN,domain,False,None,True,No missing; 2019-06-14 to 2021-10-24
5,AverageMilkFlowKgPerMin,0.0,23.5,IQR,False,Independent (172),True,Zero = likely sensor error
6,MilkFlow30To60SecondsKgPerMin,0.0,10.5,IQR,True,None,True,3155 zeros — investigate slow let-down vs sensor
7,MilkYieldFirst2MinutesKg,2.0,12.0,IQR,False,None,True,Min 2.0003 suggests sensor floor
8,TotalMilkYieldSessionKg,5.0,54.8,IQR,False,None,True,High corr with Duration; max 54.8 extreme
9,MilkingDurationSeconds,100.0,785.0,IQR,False,None,True,<150s and >600s are soft flags


In [ ]:
before = len(df_master)

# Add outlier flags to df_master
df_master["AvgFlowIsZero"] = outlier_flags["AvgFlowIsZero"]
df_master["DurationOver600s"] = outlier_flags["DurationOver600s"]

# Separate missing rows
df_missing_protected = df_master[df_master["AnimalId"].isna()].copy()
df_known_clean       = df_master[df_master["AnimalId"].notna()].copy()

known_before = len(df_known_clean)

# Drop only from known rows
df_known_clean = df_known_clean[
    ~df_known_clean["AvgFlowIsZero"] &
    ~df_known_clean["DurationOver600s"] &
    df_known_clean["AverageMilkFlowKgPerMin"].notna()
]

known_after = len(df_known_clean)

print(f"Known rows before : {known_before:,}")
print(f"Known rows dropped: {known_before - known_after:,}")
print(f"Known rows after  : {known_after:,}")
print(f"Missing rows kept : {len(df_missing_protected):,}  ← untouched")

Known rows before : 6,794,418
Known rows dropped: 463
Known rows after  : 6,793,955
Missing rows kept : 1,701,003  ← untouched


In [ ]:
# Previous code flagged NaN as invalid in lactation number, Only flag actual numbers outside 1–12.

for df in [df_known_clean, df_missing_protected]:
    df["LactationNumber_Invalid"] = (
        df["LactationNumber"].notna() &
        (~df["LactationNumber"].isin(range(1, 13)))
    )

print("Known    — invalid LactationNumber (non-NaN, outside 1-12):",
      df_known_clean["LactationNumber_Invalid"].sum())
print("Missing  — invalid LactationNumber:",
      df_missing_protected["LactationNumber_Invalid"].sum())

Known    — invalid LactationNumber (non-NaN, outside 1-12): 0
Missing  — invalid LactationNumber: 0


In [ ]:
# Duplicate Check
# A true duplicate = same animal, same date, same session, same measurements
# Check known rows only (missing rows can't be deduplicated without AnimalId)

dedup_cols = [
    "AnimalId", "EventDate", "MilkingSession",
    "TotalMilkYieldSessionKg", "MilkingDurationSeconds",
    "AverageMilkFlowKgPerMin"
]

n_total    = len(df_known_clean)
n_dupes    = df_known_clean.duplicated(subset=dedup_cols, keep=False).sum()
n_to_drop  = df_known_clean.duplicated(subset=dedup_cols, keep="first").sum()

print(f"Total known rows          : {n_total:,}")
print(f"Rows involved in duplicates: {n_dupes:,}")
print(f"Rows to drop (keep first) : {n_to_drop:,}")

# Drop duplicates
df_known_clean = df_known_clean.drop_duplicates(subset=dedup_cols, keep="first")

print(f"Known rows after dedup    : {len(df_known_clean):,}")

Total known rows          : 6,793,955
Rows involved in duplicates: 103,580
Rows to drop (keep first) : 56,591
Known rows after dedup    : 6,737,364


In [ ]:
# Rebuild df_master with clean known + protected missing
df_master = pd.concat([df_known_clean, df_missing_protected], ignore_index=True)

df_known   = df_known_clean.copy()
df_missing = df_missing_protected.copy()

print(f"df_known   : {len(df_known):,} rows")
print(f"df_missing : {len(df_missing):,} rows")
print(f"df_master  : {len(df_master):,} rows total")

# Overwrite the NEW cleaned data file
df_master.to_csv(cleaned_data_file_path, index=False)
print(f"\nOverwritten: {cleaned_data_file_path}")

df_known   : 6,737,364 rows
df_missing : 1,701,003 rows
df_master  : 8,438,367 rows total

Overwritten: /content/drive/MyDrive/MiniProject/cleaned_data.csv


In [ ]:
# Feature set & date integer
FEATURES = [
    "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds",
    "AverageMilkFlowKgPerMin",
    "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg",
    "MilkingSession",
    "EventDateInt" #date as integar for KNN
]

df_known["EventDateInt"]   = df_known["EventDate"].astype("int64") // 10**9 // 86400
df_missing["EventDateInt"] = df_missing["EventDate"].astype("int64") // 10**9 // 86400

print("Features:", FEATURES)
print("Known rows with all features present:",
      df_known[FEATURES].notna().all(axis=1).sum())

Features: ['TotalMilkYieldSessionKg', 'MilkingDurationSeconds', 'AverageMilkFlowKgPerMin', 'MilkFlow30To60SecondsKgPerMin', 'MilkYieldFirst2MinutesKg', 'MilkingSession', 'EventDateInt']
Known rows with all features present: 6737364


In [ ]:
# Phase 1: Exact Match
# Reset df_missing to clean state before rerunning
df_missing = df_master[df_master["AnimalId"].isna()].copy()
df_missing["EventDateInt"] = df_missing["EventDate"].astype("int64") // 10**9 // 86400

MATCH_COLS = ["EventDate", "MilkingSession", "TotalMilkYieldSessionKg"]

known_lookup = (
    df_known
    .groupby(MATCH_COLS)["AnimalId"]
    .nunique()
    .reset_index(name="CandidateCount")
)
known_lookup = known_lookup[known_lookup["CandidateCount"] == 1]

known_id_lookup = (
    df_known
    .groupby(MATCH_COLS)["AnimalId"]
    .first()
    .reset_index()
)
known_id_lookup = known_id_lookup[
    known_id_lookup[MATCH_COLS].apply(tuple, axis=1)
    .isin(known_lookup[MATCH_COLS].apply(tuple, axis=1))
]

df_missing = df_missing.merge(
    known_id_lookup.rename(columns={"AnimalId": "AnimalId_ExactMatch"}),
    on=MATCH_COLS,
    how="left"
)

exact_resolved   = df_missing["AnimalId_ExactMatch"].notna().sum()
exact_unresolved = df_missing["AnimalId_ExactMatch"].isna().sum()

print(f"Exact match resolved  : {exact_resolved:,}")
print(f"Still unresolved      : {exact_unresolved:,}")
print(f"Resolution rate       : {exact_resolved/len(df_missing)*100:.1f}%")

Exact match resolved  : 47,781
Still unresolved      : 1,653,222
Resolution rate       : 2.8%


In [ ]:
# How are the missing rows distributed across time?
df_missing["EventDate"].dt.to_period("M").value_counts().sort_index()

,count
EventDate,
2019-06,28601
2019-07,50324
2019-08,34453
2019-09,61442
2019-10,58486
2019-11,48665
2019-12,57259
2020-01,54044
2020-02,58843


In [ ]:
# Build validation and training pool
FEATURES_SPATIAL = [
    "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds",
    "AverageMilkFlowKgPerMin",
    "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg",
    "MilkingSession"
]

# 10% of each animal's records held out for validation
val_sample = df_known.groupby("AnimalId", group_keys=False).apply(
    lambda x: x.sample(max(1, int(len(x) * 0.10)), random_state=42)
)
train_pool = df_known.drop(val_sample.index)

# Cap validation at 50K rows so it runs in reasonable time on Colab
val_sample = val_sample.sample(min(50_000, len(val_sample)), random_state=42)

print(f"Training pool     : {len(train_pool):,}")
print(f"Validation sample : {len(val_sample):,}")
print(f"Unique animals in train: {train_pool['AnimalId'].nunique():,}")

/tmp/ipykernel_749/94190658.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_sample = df_known.groupby("AnimalId", group_keys=False).apply(


Training pool     : 6,067,580
Validation sample : 50,000
Unique animals in train: 9,082


In [ ]:
# ── Windowed KNN validation — vectorized by date ──────────────────────────────
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import numpy as np

WINDOW_DAYS = 30
K = 5
results = []
skipped = 0

# Group validation rows by EventDate so we fit one KNN per unique date
# instead of one KNN per row
unique_dates = val_sample["EventDateInt"].unique()
print(f"Unique dates to process: {len(unique_dates):,}")

for i, date_int in enumerate(unique_dates):
    if i % 100 == 0:
        print(f"  Processing date {i:,} / {len(unique_dates):,}...")

    # All validation rows on this date
    val_rows = val_sample[val_sample["EventDateInt"] == date_int]

    # Training window: ±30 days
    window = train_pool[
        (train_pool["EventDateInt"] >= date_int - WINDOW_DAYS) &
        (train_pool["EventDateInt"] <= date_int + WINDOW_DAYS)
    ]

    if len(window) < K or window["AnimalId"].nunique() < 2:
        skipped += len(val_rows)
        for idx in val_rows.index:
            results.append({"true": val_rows.loc[idx, "AnimalId"],
                            "pred": None, "confidence": 0})
        continue

    # Fit ONE KNN for this date window — covers all val rows on this date
    X_window = window[FEATURES_SPATIAL].dropna()
    y_window = window.loc[X_window.index, "AnimalId"]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_window)

    knn = KNeighborsClassifier(
        n_neighbors=min(K, len(X_window)),
        metric="euclidean"
    )
    knn.fit(X_scaled, y_window)

    # Predict ALL validation rows on this date at once
    X_val_rows = val_rows[FEATURES_SPATIAL].dropna()
    if len(X_val_rows) == 0:
        continue

    X_val_scaled = scaler.transform(X_val_rows.values)
    preds  = knn.predict(X_val_scaled)
    probas = knn.predict_proba(X_val_scaled).max(axis=1)

    for j, idx in enumerate(X_val_rows.index):
        results.append({
            "true":       val_rows.loc[idx, "AnimalId"],
            "pred":       preds[j],
            "confidence": probas[j]
        })

print("Done.")

Unique dates to process: 830
  Processing date 0 / 830...


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

  Processing date 100 / 830...


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

  Processing date 200 / 830...


...

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

  Processing date 800 / 830...


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

Done.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
#Validation results
import pandas as pd

results_df = pd.DataFrame(results)
resolved   = results_df["pred"].notna()
correct    = (results_df["true"] == results_df["pred"]).sum()
total      = resolved.sum()

print(f"Windowed KNN validation (K={K}, window=±{WINDOW_DAYS} days)")
print(f"Correct            : {correct:,}")
print(f"Incorrect          : {total - correct:,}")
print(f"Skipped (too small): {skipped:,}")
print(f"Accuracy           : {correct/total*100:.2f}%")

print(f"\nConfidence distribution:")
display(
    results_df[resolved]["confidence"]
    .describe()
    .to_frame()
)

Windowed KNN validation (K=5, window=±30 days)
Correct            : 996
Incorrect          : 49,004
Skipped (too small): 0
Accuracy           : 1.99%

Confidence distribution:


,confidence
count,50000.000000
mean,0.225580
std,0.073055
min,0.200000
25%,0.200000
50%,0.200000
75%,0.200000
max,1.000000


In [ ]:
# How many missing records per date on average?
missing_per_date = df_missing.groupby("EventDate").size()
print(missing_per_date.describe())
print(f"\nMissing records always divisible by 3 (session check):")
print((missing_per_date % 3 == 0).all())

# How many unique animals are active (have known records) on each date?
known_per_date = df_known.groupby("EventDate")["AnimalId"].nunique()
print(f"\nKnown animals per date:")
print(known_per_date.describe())

count     830.000000
mean     2049.401205
std       344.108206
min        21.000000
25%      1933.500000
50%      2106.500000
75%      2199.000000
max      5716.000000
dtype: float64

Missing records always divisible by 3 (session check):
False

Known animals per date:
count     830.000000
mean     3246.574699
std       440.798621
min        47.000000
25%      3072.000000
50%      3377.500000
75%      3516.000000
max      3801.000000
Name: AnimalId, dtype: float64


In [ ]:
# Are the SAME animals missing every day, or different animals each day?

# For each date, how many missing records are there per session?
missing_per_date_session = (
    df_missing.groupby(["EventDate", "MilkingSession"])
    .size()
    .unstack(fill_value=0)
)
print("Missing records per session per date (first 10 dates):")
display(missing_per_date_session.head(10))

# Check: on a given date, do the missing counts match across sessions?
missing_per_date_session["balanced"] = (
    missing_per_date_session[1] == missing_per_date_session[2]
) & (
    missing_per_date_session[2] == missing_per_date_session[3]
)
print(f"\nDates where missing count is equal across all 3 sessions: "
      f"{missing_per_date_session['balanced'].sum()} / {len(missing_per_date_session)}")

# Key question: does missing count per date ≈ 20% of active animals?
comparison = pd.DataFrame({
    "KnownAnimals":   known_per_date,
    "MissingRecords": missing_per_date,
    "MissingPerSession": missing_per_date / 3
})
comparison["MissingPct"] = (
    comparison["MissingPerSession"] / comparison["KnownAnimals"] * 100
).round(1)
print("\nMissing % relative to known animals per date:")
display(comparison["MissingPct"].describe().to_frame())

Missing records per session per date (first 10 dates):


MilkingSession,1,2,3
EventDate,,,
2019-06-14,196,200,198
2019-06-15,624,686,637
2019-06-16,702,752,705
2019-06-17,711,702,713
2019-06-18,692,706,695
2019-06-19,668,658,714
2019-06-20,714,670,748
2019-06-21,725,715,740
2019-06-22,705,681,684



Dates where missing count is equal across all 3 sessions: 0 / 830

Missing % relative to known animals per date:


,MissingPct
count,830.000000
mean,21.030482
std,2.234389
min,14.900000
25%,20.400000
50%,20.700000
75%,21.100000
max,65.800000


In [ ]:
# Are the same animals missing every day?

# For known animals: which ones appear consistently across the full date range?
animal_date_counts = df_known.groupby("AnimalId")["EventDate"].nunique()
total_dates = df_known["EventDate"].nunique()

print(f"Total unique dates in dataset: {total_dates}")
print(f"\nAnimal date coverage:")
display(animal_date_counts.describe().to_frame())

# Animals present on almost every date (>90% of dates)
always_present = animal_date_counts[animal_date_counts >= total_dates * 0.90]
sometimes_present = animal_date_counts[
    (animal_date_counts >= total_dates * 0.50) &
    (animal_date_counts < total_dates * 0.90)
]
rarely_present = animal_date_counts[animal_date_counts < total_dates * 0.50]

print(f"\nAnimals present on >90% of dates  : {len(always_present):,}")
print(f"Animals present on 50-90% of dates: {len(sometimes_present):,}")
print(f"Animals present on <50% of dates  : {len(rarely_present):,}")

# Key question: does known animal count + missing animal count ≈ total herd?
avg_known_per_date   = known_per_date.mean()
avg_missing_per_date = (missing_per_date / 3).mean()
print(f"\nAvg known animals per date   : {avg_known_per_date:.0f}")
print(f"Avg missing animals per date : {avg_missing_per_date:.0f}")
print(f"Total estimated herd size    : {avg_known_per_date + avg_missing_per_date:.0f}")
print(f"Known unique AnimalIds       : {df_known['AnimalId'].nunique():,}")

Total unique dates in dataset: 830

Animal date coverage:


,EventDate
count,9087.000000
mean,296.539782
std,195.888553
min,1.000000
25%,123.000000
50%,279.000000
75%,471.000000
max,701.000000



Animals present on >90% of dates  : 0
Animals present on 50-90% of dates: 2,881
Animals present on <50% of dates  : 6,206

Avg known animals per date   : 3247
Avg missing animals per date : 683
Total estimated herd size    : 3930
Known unique AnimalIds       : 9,087


In [ ]:
# ── Who was EXPECTED to be present but is missing on each date? ───────────────

# Build each animal's active window: first to last known record date
animal_active_window = df_known.groupby("AnimalId")["EventDate"].agg(
    ActiveFrom="min",
    ActiveTo="max"
)

# For a given date, an animal is "expected" if date falls within their window
# Sample check on one date first
sample_date = pd.Timestamp("2020-06-15")

expected_animals = animal_active_window[
    (animal_active_window["ActiveFrom"] <= sample_date) &
    (animal_active_window["ActiveTo"]   >= sample_date)
]

actually_present = df_known[df_known["EventDate"] == sample_date]["AnimalId"].unique()

expected_not_present = expected_animals[
    ~expected_animals.index.isin(actually_present)
]

print(f"Date: {sample_date.date()}")
print(f"Expected active animals : {len(expected_animals):,}")
print(f"Actually present (known): {len(actually_present):,}")
print(f"Missing records on date : {(df_missing['EventDate'] == sample_date).sum():,}")
print(f"Expected but absent     : {len(expected_not_present):,}")
print(f"\nDo absent animals \u2248 missing records / 3?")
missing_on_date = (df_missing["EventDate"] == sample_date).sum()
print(f"Missing records: {missing_on_date:,}, Expected but absent: {len(expected_not_present):,}")

Date: 2020-06-15
Expected active animals : 4,690
Actually present (known): 3,126
Missing records on date : 1,892
Expected but absent     : 1,564

Do absent animals ≈ missing records / 3?
Missing records: 1,892, Expected but absent: 1,564


In [ ]:
# ── Tighter active window using rolling presence ──────────────────────────────
# An animal is "active" on a date only if they have known records
# within ±15 days of that date (not just within their full career span)

sample_date = pd.Timestamp("2020-06-15")
ACTIVE_WINDOW = 15  # days

# Animals with a known record within ±15 days of sample date
nearby_known = df_known[
    (df_known["EventDate"] >= sample_date - pd.Timedelta(days=ACTIVE_WINDOW)) &
    (df_known["EventDate"] <= sample_date + pd.Timedelta(days=ACTIVE_WINDOW))
]

recently_active = nearby_known["AnimalId"].unique()
actually_present = df_known[
    df_known["EventDate"] == sample_date
]["AnimalId"].unique()

candidate_pool = [a for a in recently_active if a not in actually_present]
missing_on_date = (df_missing["EventDate"] == sample_date).sum()

print(f"Date                         : {sample_date.date()}")
print(f"Recently active (±{ACTIVE_WINDOW} days)    : {len(recently_active):,}")
print(f"Actually present today       : {len(actually_present):,}")
print(f"Candidate pool (absent)      : {len(candidate_pool):,}")
print(f"Missing records on date      : {missing_on_date:,}")
print(f"Missing animals (records/3)  : {missing_on_date/3:.0f}")
print(f"\nCandidate pool vs missing animals ratio: {len(candidate_pool)/(missing_on_date/3):.2f}x")

# Try different window sizes
print("\n── Sensitivity to window size ──")
for w in [7, 15, 30, 45]:
    nearby = df_known[
        (df_known["EventDate"] >= sample_date - pd.Timedelta(days=w)) &
        (df_known["EventDate"] <= sample_date + pd.Timedelta(days=w))
    ]["AnimalId"].unique()
    candidates = len([a for a in nearby if a not in actually_present])
    print(f"  Window ±{w:2d} days → recently active: {len(nearby):,}  "
          f"candidates: {candidates:,}  "
          f"vs missing animals: {missing_on_date/3:.0f}")

Date                         : 2020-06-15
Recently active (±15 days)    : 4,471
Actually present today       : 3,126
Candidate pool (absent)      : 1,345
Missing records on date      : 1,892
Missing animals (records/3)  : 631

Candidate pool vs missing animals ratio: 2.13x

── Sensitivity to window size ──
  Window ± 7 days → recently active: 4,224  candidates: 1,098  vs missing animals: 631
  Window ±15 days → recently active: 4,471  candidates: 1,345  vs missing animals: 631
  Window ±30 days → recently active: 4,837  candidates: 1,711  vs missing animals: 631
  Window ±45 days → recently active: 5,053  candidates: 1,927  vs missing animals: 631


In [ ]:
# Phase 3: Fill remaining identifier columns
# For each recovered AnimalId, look up the known record closest in EventDate
# and fill LactationNumber, DaysInMilk, ReproductionStatus from it

known_ref = df_known[
    ["AnimalId", "EventDate", "LactationNumber", "DaysInMilk", "ReproductionStatus"]
].dropna()

def fill_identifiers(missing_row, known_ref):
    animal_records = known_ref[known_ref["AnimalId"] == missing_row["AnimalId_Recovered"]]
    if animal_records.empty:
        return pd.Series({
            "LactationNumber": np.nan,
            "DaysInMilk": np.nan,
            "ReproductionStatus": np.nan
        })
    # Find closest record by date
    date_diffs = (animal_records["EventDate"] - missing_row["EventDate"]).abs()
    closest = animal_records.loc[date_diffs.idxmin()]
    return pd.Series({
        "LactationNumber":   closest["LactationNumber"],
        "DaysInMilk":        closest["DaysInMilk"],
        "ReproductionStatus": closest["ReproductionStatus"]
    })

# Combine exact match + KNN into one recovered AnimalId column
df_missing["AnimalId_Recovered"] = df_missing["AnimalId_ExactMatch"].fillna(
    df_unresolved_clean["AnimalId_KNN"].reindex(df_missing.index)
)

# Apply fill (this will take a few minutes on 1.7M rows)
print("Filling identifier columns — this may take a few minutes...")
filled = df_missing.apply(fill_identifiers, axis=1, known_ref=known_ref)
df_missing[["LactationNumber_Filled", "DaysInMilk_Filled", "ReproductionStatus_Filled"]] = filled

print("Done.")
print(f"LactationNumber filled    : {df_missing['LactationNumber_Filled'].notna().sum():,}")
print(f"DaysInMilk filled         : {df_missing['DaysInMilk_Filled'].notna().sum():,}")
print(f"ReproductionStatus filled : {df_missing['ReproductionStatus_Filled'].notna().sum():,}")

In [ ]:
# Export Final Dataset
# Overwrite original file with recovered identifiers
df_missing_final = df_missing.copy()
df_missing_final["AnimalId"]           = df_missing_final["AnimalId_Recovered"]
df_missing_final["LactationNumber"]    = df_missing_final["LactationNumber_Filled"]
df_missing_final["DaysInMilk"]         = df_missing_final["DaysInMilk_Filled"]
df_missing_final["ReproductionStatus"] = df_missing_final["ReproductionStatus_Filled"]

# Drop working columns
df_missing_final = df_missing_final.drop(columns=[
    "AnimalId_ExactMatch", "AnimalId_Recovered",
    "LactationNumber_Filled", "DaysInMilk_Filled", "ReproductionStatus_Filled"
], errors="ignore")

df_final = pd.concat([df_known, df_missing_final], ignore_index=True)
df_final = df_final.sort_values(["AnimalId", "EventDate", "MilkingSession"]).reset_index(drop=True)

df_final.to_csv(data_file_path, index=False)

still_missing = df_final["AnimalId"].isna().sum()
print(f"Final rows          : {len(df_final):,}")
print(f"AnimalId still null : {still_missing:,}")
print(f"Saved to            : {data_file_path}")

In [ ]:
# ── Check: do total sessions per date add up correctly? ──────────────────────
# If known + missing = total expected (herd × 3 sessions),
# we can use session-level balancing

sample_dates = df_known["EventDate"].unique()[:30]  # check first 30 dates

balance_check = []
for date in sorted(sample_dates):
    known_count   = (df_known["EventDate"] == date).sum()
    missing_count = (df_missing["EventDate"] == date).sum()
    total         = known_count + missing_count

    # How many unique animals are known on this date?
    known_animals = df_known[df_known["EventDate"] == date]["AnimalId"].nunique()

    balance_check.append({
        "Date":          date.date(),
        "KnownRecords":  known_count,
        "MissingRecords": missing_count,
        "TotalRecords":  total,
        "KnownAnimals":  known_animals,
        "TotalRecords/3": round(total / 3, 1),
        "BalancedHerd":  round(total / 3) == known_animals + round(missing_count/3)
    })

df_balance = pd.DataFrame(balance_check)
display(df_balance.head(20))

print(f"\nTotal records always divisible by 3: {(df_balance['TotalRecords'] % 3 == 0).all()}")
print(f"Dates where known+missing = balanced herd×3: {df_balance['BalancedHerd'].sum()} / {len(df_balance)}")

,Date,KnownRecords,MissingRecords,TotalRecords,KnownAnimals,TotalRecords/3,BalancedHerd
0,2019-06-16,8551,2159,10710,3424,3570.0,False
1,2019-08-18,7463,1800,9263,3030,3087.7,False
2,2019-11-06,7286,1827,9113,2857,3037.7,False
3,2019-12-04,6170,1584,7754,2495,2584.7,False
4,2019-12-09,6675,1652,8327,2716,2775.7,False
5,2019-12-14,6366,1598,7964,2611,2654.7,False
6,2020-01-08,6514,1658,8172,2561,2724.0,False
7,2020-01-21,6974,1654,8628,2807,2876.0,False
8,2020-02-01,8396,2134,10530,3403,3510.0,False
9,2020-02-03,8223,1995,10218,3326,3406.0,False



Total records always divisible by 3: False
Dates where known+missing = balanced herd×3: 0 / 30


In [ ]:
# Do missing records cluster by animal across days?
# If the same "ghost" records appear consistently, they should show
# similar yield patterns across consecutive days

# Pick one week of missing records
week_missing = df_missing[
    (df_missing["EventDate"] >= "2020-06-10") &
    (df_missing["EventDate"] <= "2020-06-17")
].copy()

# Pick same week of known records
week_known = df_known[
    (df_known["EventDate"] >= "2020-06-10") &
    (df_known["EventDate"] <= "2020-06-17")
].copy()

print(f"Missing records in week : {len(week_missing):,}")
print(f"Known records in week   : {len(week_known):,}")
print(f"Known animals in week   : {week_known['AnimalId'].nunique():,}")

# How variable is yield day-to-day for a known animal?
# Low variance = yield is a stable fingerprint
yield_cv = (
    week_known.groupby("AnimalId")["TotalMilkYieldSessionKg"]
    .agg(["mean", "std"])
    .assign(CV=lambda x: x["std"] / x["mean"] * 100)
    .dropna()
)

print(f"\nCoefficient of variation of yield across the week (known animals):")
display(yield_cv["CV"].describe().to_frame())
print("\nLow CV = yield is stable and distinctive across days")
print("High CV = yield fluctuates too much to use as fingerprint")

Missing records in week : 16,269
Known records in week   : 66,399
Known animals in week   : 4,049

Coefficient of variation of yield across the week (known animals):


,CV
count,4039.000000
mean,8.893564
std,4.195322
min,0.000000
25%,6.251807
50%,7.848252
75%,10.280806
max,45.501627



Low CV = yield is stable and distinctive across days
High CV = yield fluctuates too much to use as fingerprint


In [ ]:
# Build 7-day rolling profiles for known animals
PROFILE_FEATURES = [
    "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds",
    "AverageMilkFlowKgPerMin",
    "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg"
]

# For each animal, compute rolling 7-day mean per feature
# This gives a stable "fingerprint" per animal per time period
known_profiles = (
    df_known
    .sort_values(["AnimalId", "EventDate"])
    .groupby("AnimalId")[PROFILE_FEATURES]
    .transform(lambda x: x.rolling(7, min_periods=3).mean())
)
known_profiles["AnimalId"]    = df_known["AnimalId"].values
known_profiles["EventDate"]   = df_known["EventDate"].values
known_profiles["EventDateInt"] = df_known["EventDateInt"].values
known_profiles = known_profiles.dropna(subset=PROFILE_FEATURES)

print(f"Known profiles built : {len(known_profiles):,} rows")
print(f"Animals covered      : {known_profiles['AnimalId'].nunique():,}")

Known profiles built : 6,719,195 rows
Animals covered      : 9,087


In [ ]:
# Validation: profile-based KNN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import numpy as np

K = 5
WINDOW_DAYS = 15
results_profile = []
skipped_profile = 0

# Fix: use include_groups=False and reset_index cleanly
val_profile = (
    known_profiles
    .groupby("AnimalId", group_keys=False)
    .apply(lambda x: x.sample(max(1, int(len(x) * 0.10)), random_state=42),
           include_groups=False)
)

# AnimalId is now missing from val_profile (excluded by include_groups=False)
# Add it back from known_profiles using the index
val_profile["AnimalId"] = known_profiles.loc[val_profile.index, "AnimalId"]
val_profile = val_profile.sample(min(30_000, len(val_profile)), random_state=42)

train_profile = known_profiles.drop(val_profile.index)

print(f"Profile train rows : {len(train_profile):,}")
print(f"Profile val rows   : {len(val_profile):,}")
print(f"Animals in val     : {val_profile['AnimalId'].nunique():,}")

# Vectorized by date
unique_dates = val_profile["EventDateInt"].unique()
print(f"Unique dates       : {len(unique_dates):,}")

for i, date_int in enumerate(unique_dates):
    if i % 100 == 0:
        print(f"  {i:,} / {len(unique_dates):,} dates...")

    val_rows = val_profile[val_profile["EventDateInt"] == date_int]

    window = train_profile[
        (train_profile["EventDateInt"] >= date_int - WINDOW_DAYS) &
        (train_profile["EventDateInt"] <= date_int + WINDOW_DAYS)
    ]

    if len(window) < K or window["AnimalId"].nunique() < 2:
        skipped_profile += len(val_rows)
        for idx in val_rows.index:
            results_profile.append({
                "true": val_rows.loc[idx, "AnimalId"],
                "pred": None, "confidence": 0
            })
        continue

    X_window = window[PROFILE_FEATURES].values
    y_window = window["AnimalId"].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_window)

    knn = KNeighborsClassifier(
        n_neighbors=min(K, len(X_window)),
        metric="euclidean",
        n_jobs=-1
    )
    knn.fit(X_scaled, y_window)

    X_val_scaled = scaler.transform(val_rows[PROFILE_FEATURES].values)
    preds  = knn.predict(X_val_scaled)
    probas = knn.predict_proba(X_val_scaled).max(axis=1)

    for j, idx in enumerate(val_rows.index):
        results_profile.append({
            "true":       val_rows.loc[idx, "AnimalId"],
            "pred":       preds[j],
            "confidence": probas[j]
        })

print("Done.")

Profile train rows : 6,689,195
Profile val rows   : 30,000
Animals in val     : 7,418
Unique dates       : 830
  0 / 830 dates...
  100 / 830 dates...
  200 / 830 dates...
  300 / 830 dates...
  400 / 830 dates...
  500 / 830 dates...
  600 / 830 dates...
  700 / 830 dates...
  800 / 830 dates...
Done.


In [ ]:
# Profile KNN validation results
results_profile_df = pd.DataFrame(results_profile)
resolved  = results_profile_df["pred"].notna()
correct   = (results_profile_df["true"] == results_profile_df["pred"]).sum()
total     = resolved.sum()

print(f"Profile KNN Validation (K={K}, window=±{WINDOW_DAYS} days)")
print(f"Correct            : {correct:,}")
print(f"Incorrect          : {total - correct:,}")
print(f"Skipped            : {skipped_profile:,}")
print(f"Accuracy           : {correct/total*100:.2f}%")
print(f"\nConfidence distribution:")
display(results_profile_df[resolved]["confidence"].describe().to_frame())

Profile KNN Validation (K=5, window=±15 days)
Correct            : 8
Incorrect          : 29,992
Skipped            : 0
Accuracy           : 0.03%

Confidence distribution:


,confidence
count,30000.000000
mean,0.200573
std,0.010693
min,0.200000
25%,0.200000
50%,0.200000
75%,0.200000
max,0.400000


In [ ]:
# Does DaysInMilk + Yield create separable clusters?
# Check how many animals share the same DaysInMilk on a given date
sample_date = pd.Timestamp("2020-06-15")

known_on_date = df_known[df_known["EventDate"] == sample_date][
    ["AnimalId", "DaysInMilk", "TotalMilkYieldSessionKg",
     "LactationNumber", "MilkingSession"]
].dropna()

print(f"Known animals on {sample_date.date()}: {known_on_date['AnimalId'].nunique():,}")
print(f"\nDaysInMilk distribution:")
display(known_on_date["DaysInMilk"].describe().to_frame())

# How unique is DaysInMilk per animal on this date?
dim_counts = known_on_date.groupby("DaysInMilk")["AnimalId"].nunique()
print(f"\nDaysInMilk values shared by >1 animal: {(dim_counts > 1).sum()}")
print(f"DaysInMilk values unique to 1 animal : {(dim_counts == 1).sum()}")

# Combine DaysInMilk + LactationNumber as identifier
known_on_date["DIM_Lac_Key"] = (
    known_on_date["DaysInMilk"].astype(str) + "_" +
    known_on_date["LactationNumber"].astype(str)
)
key_counts = known_on_date.groupby("DIM_Lac_Key")["AnimalId"].nunique()
print(f"\nDIM+Lactation combos unique to 1 animal: {(key_counts == 1).sum()}")
print(f"DIM+Lactation combos shared by >1 animal: {(key_counts > 1).sum()}")
print(f"\nIf unique, % of animals identifiable by DIM+Lactation alone: "
      f"{(key_counts == 1).sum() / known_on_date['AnimalId'].nunique() * 100:.1f}%")

Known animals on 2020-06-15: 3,126

DaysInMilk distribution:


,DaysInMilk
count,7661.000000
mean,158.608276
std,104.632215
min,2.000000
25%,73.000000
50%,146.000000
75%,232.000000
max,604.000000



DaysInMilk values shared by >1 animal: 370
DaysInMilk values unique to 1 animal : 60

DIM+Lactation combos unique to 1 animal: 629
DIM+Lactation combos shared by >1 animal: 780

If unique, % of animals identifiable by DIM+Lactation alone: 20.1%


In [ ]:
# Estimate true number of ghost animals
# If the same ~700 animals are missing every day across 830 days,
# total missing records / 830 days / 3 sessions = number of ghost animals

total_missing   = len(df_missing)
avg_per_session = total_missing / 830 / 3

print(f"Total missing records    : {total_missing:,}")
print(f"Estimated ghost animals  : {avg_per_session:.0f}")

# Cross-check: how many animals are in known but appear on fewer dates
# than expected? These may be animals whose missing records we can find
known_date_counts = df_known.groupby("AnimalId")["EventDate"].nunique()
sparse_animals = known_date_counts[known_date_counts < 100]
print(f"\nKnown animals with <100 active dates : {len(sparse_animals):,}")
print(f"These may have missing records filling the gap")

# What's the total missing dates for these sparse animals?
print(f"\nExpected dates per animal if active full period: 830")
print(f"Mean dates per known animal: {known_date_counts.mean():.0f}")
print(f"Gap per animal: {830 - known_date_counts.mean():.0f} days missing on average")
print(f"Gap × 9087 animals × 3 sessions = {(830 - known_date_counts.mean()) * 9087 * 3:,.0f}")
print(f"Actual missing records            = {total_missing:,}")

Total missing records    : 1,701,003
Estimated ghost animals  : 683

Known animals with <100 active dates : 1,927
These may have missing records filling the gap

Expected dates per animal if active full period: 830
Mean dates per known animal: 297
Gap per animal: 533 days missing on average
Gap × 9087 animals × 3 sessions = 14,542,659
Actual missing records            = 1,701,003


In [ ]:
# Validate clustering hypothesis
# If 683 ghost animals each milk 3x/day for ~830 days:
# 683 × 3 × 830 = expected missing records

estimated_animals = 683
expected = estimated_animals * 3 * 830
actual   = len(df_missing)

print(f"Estimated ghost animals      : {estimated_animals}")
print(f"Expected missing if all days : {expected:,}")
print(f"Actual missing records       : {actual:,}")
print(f"Coverage                     : {actual/expected*100:.1f}%")

# So ghost animals are present on what % of days?
days_per_ghost = actual / estimated_animals / 3
print(f"\nAvg days each ghost appears  : {days_per_ghost:.0f} / 830")
print(f"Coverage per ghost animal    : {days_per_ghost/830*100:.1f}%")

# Check: do missing records per date stay stable? (confirms fixed ghost herd)
missing_per_date = df_missing.groupby("EventDate").size() / 3
print(f"\nGhost animals per date:")
print(f"  Mean : {missing_per_date.mean():.0f}")
print(f"  Std  : {missing_per_date.std():.0f}")
print(f"  Min  : {missing_per_date.min():.0f}")
print(f"  Max  : {missing_per_date.max():.0f}")

Estimated ghost animals      : 683
Expected missing if all days : 1,700,670
Actual missing records       : 1,701,003
Coverage                     : 100.0%

Avg days each ghost appears  : 830 / 830
Coverage per ghost animal    : 100.0%

Ghost animals per date:
  Mean : 683
  Std  : 115
  Min  : 7
  Max  : 1905


In [ ]:
# K-Means clustering of missing records into 683 ghost animals

N_CLUSTERS = 683

CLUSTER_FEATURES = [
    "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds",
    "AverageMilkFlowKgPerMin",
    "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg"
]

# Prepare missing records
df_missing_model = df_missing[CLUSTER_FEATURES].dropna().copy()

print(f"Missing records for clustering : {len(df_missing_model):,}")
print(f"Target clusters                : {N_CLUSTERS}")

# Scale features
scaler_cluster = StandardScaler()
X_missing_scaled = scaler_cluster.fit_transform(df_missing_model.values)

# MiniBatchKMeans — much faster than KMeans on 1.7M rows
print("\nRunning MiniBatchKMeans...")
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    batch_size=10_000,
    n_init=10,
    max_iter=300,
    verbose=1
)
kmeans.fit(X_missing_scaled)

df_missing_model["Cluster"] = kmeans.labels_
df_missing.loc[df_missing_model.index, "Cluster"] = kmeans.labels_

print(f"\nClustering done.")
print(f"Cluster label range: {kmeans.labels_.min()} to {kmeans.labels_.max()}")
print(f"\nCluster size distribution:")
cluster_sizes = pd.Series(kmeans.labels_).value_counts()
display(cluster_sizes.describe().to_frame())

Missing records for clustering : 1,700,962
Target clusters                : 683

Running MiniBatchKMeans...
Init 1/10 with method k-means++
Inertia for init 1/10: 3049.6205165643432
Init 2/10 with method k-means++
Inertia for init 2/10: 3079.5376239070424
Init 3/10 with method k-means++
Inertia for init 3/10: 3159.621965148532
Init 4/10 with method k-means++
Inertia for init 4/10: 3019.789775495192
Init 5/10 with method k-means++
Inertia for init 5/10: 3025.996349955724
Init 6/10 with method k-means++
Inertia for init 6/10: 3008.8446681277433
Init 7/10 with method k-means++
Inertia for init 7/10: 3202.6580709383666
Init 8/10 with method k-means++
Inertia for init 8/10: 4019.1806963638296
Init 9/10 with method k-means++
Inertia for init 9/10: 3999.9177840639295
Init 10/10 with method k-means++
Inertia for init 10/10: 3079.3836715209586
[MiniBatchKMeans] Reassigning 33 cluster centers.
Minibatch step 1/51028: mean batch inertia: 0.10729958306476274
Minibatch step 2/51028: mean batch iner

,count
count,683.000000
mean,2490.427526
std,1338.446394
min,73.000000
25%,1598.000000
50%,2256.000000
75%,3337.500000
max,6963.000000


In [ ]:
# ── Match each cluster centroid to a known animal ─────────────────────────────
from sklearn.neighbors import NearestNeighbors

# Build known animal profiles — median per animal across all records
known_animal_profiles = (
    df_known
    .groupby("AnimalId")[CLUSTER_FEATURES]
    .median()
    .dropna()
)

print(f"Known animal profiles built : {len(known_animal_profiles):,}")

# Scale known profiles using same scaler
X_known_profiles = scaler_cluster.transform(known_animal_profiles.values)

# Get cluster centroids (already in scaled space)
centroids = kmeans.cluster_centers_

# Match each centroid to nearest known animal profile
nn = NearestNeighbors(n_neighbors=1, metric="euclidean", n_jobs=-1)
nn.fit(X_known_profiles)

distances, indices = nn.kneighbors(centroids)

# Build cluster → AnimalId mapping
cluster_to_animal = pd.DataFrame({
    "Cluster":          range(N_CLUSTERS),
    "AnimalId_Cluster": known_animal_profiles.index[indices.flatten()],
    "MatchDistance":    distances.flatten()
})

print(f"\nCluster → Animal mapping built")
print(f"\nMatch distance distribution:")
display(cluster_to_animal["MatchDistance"].describe().to_frame())

unique_animals_assigned = cluster_to_animal["AnimalId_Cluster"].nunique()
print(f"\nUnique animals assigned     : {unique_animals_assigned:,} / {N_CLUSTERS}")
print(f"Duplicate animal assignments: {N_CLUSTERS - unique_animals_assigned:,}")

Known animal profiles built : 9,087

Cluster → Animal mapping built

Match distance distribution:


,MatchDistance
count,683.000000
mean,0.319859
std,0.540975
min,0.031954
25%,0.105869
50%,0.163481
75%,0.287229
max,5.409545



Unique animals assigned     : 622 / 683
Duplicate animal assignments: 61


In [ ]:
# ── Resolve duplicate cluster assignments ─────────────────────────────────────
# For animals assigned to 2 clusters, keep the closest match
# and reassign the further cluster to its next nearest neighbour

# Find which animals are duplicated
duplicate_animals = (
    cluster_to_animal["AnimalId_Cluster"]
    .value_counts()
    [lambda x: x > 1]
    .index.tolist()
)
print(f"Animals assigned to multiple clusters: {len(duplicate_animals)}")

# For duplicates, find next nearest neighbour
nn_extended = NearestNeighbors(n_neighbors=10, metric="euclidean", n_jobs=-1)
nn_extended.fit(X_known_profiles)
distances_ext, indices_ext = nn_extended.kneighbors(centroids)

# Resolve greedily: process clusters by distance (best matches first)
cluster_to_animal_resolved = cluster_to_animal.copy()
assigned_animals = set()
final_mapping = {}

# Sort all clusters by match distance ascending — assign best matches first
sorted_clusters = cluster_to_animal.sort_values("MatchDistance")

for _, row in sorted_clusters.iterrows():
    cluster_id = int(row["Cluster"])

    # Try neighbours in order until we find an unassigned animal
    for rank in range(10):
        candidate_animal = known_animal_profiles.index[indices_ext[cluster_id, rank]]
        candidate_dist   = distances_ext[cluster_id, rank]

        if candidate_animal not in assigned_animals:
            final_mapping[cluster_id] = {
                "AnimalId_Cluster": candidate_animal,
                "MatchDistance":    candidate_dist,
                "MatchRank":        rank  # 0 = first choice, 1 = second choice, etc.
            }
            assigned_animals.add(candidate_animal)
            break

df_final_mapping = pd.DataFrame.from_dict(final_mapping, orient="index")
df_final_mapping.index.name = "Cluster"
df_final_mapping = df_final_mapping.reset_index()

print(f"\nFinal unique assignments    : {df_final_mapping['AnimalId_Cluster'].nunique():,}")
print(f"First choice assignments    : {(df_final_mapping['MatchRank'] == 0).sum():,}")
print(f"Fallback assignments (rank>0): {(df_final_mapping['MatchRank'] > 0).sum():,}")
print(f"\nMatch distance after dedup:")
display(df_final_mapping["MatchDistance"].describe().to_frame())

Animals assigned to multiple clusters: 34

Final unique assignments    : 670
First choice assignments    : 618
Fallback assignments (rank>0): 52

Match distance after dedup:


,MatchDistance
count,670.000000
mean,0.296149
std,0.457764
min,0.031954
25%,0.105400
50%,0.160874
75%,0.281633
max,3.677991


In [ ]:
# ── Assign recovered AnimalId to df_missing ───────────────────────────────────
df_missing["AnimalId_Cluster"] = df_missing["Cluster"].map(
    df_final_mapping.set_index("Cluster")["AnimalId_Cluster"]
)

# Combine with exact match from Phase 1
df_missing["AnimalId_Recovered"] = (
    df_missing["AnimalId_ExactMatch"]
    .fillna(df_missing["AnimalId_Cluster"])
)

df_missing["RecoveryMethod"] = "Unresolved"
df_missing.loc[
    df_missing["AnimalId_ExactMatch"].notna(), "RecoveryMethod"
] = "ExactMatch"
df_missing.loc[
    df_missing["AnimalId_ExactMatch"].isna() &
    df_missing["AnimalId_Cluster"].notna(), "RecoveryMethod"
] = "Clustering"

print("=== Recovery Summary ===")
display(df_missing["RecoveryMethod"].value_counts().to_frame())
print(f"\nTotal recovered  : {df_missing['AnimalId_Recovered'].notna().sum():,}")
print(f"Total unresolved : {df_missing['AnimalId_Recovered'].isna().sum():,}")

# ── Fill LactationNumber, DaysInMilk, ReproductionStatus ─────────────────────
# Look up each recovered animal's known records and fill by nearest date
known_ref = df_known[
    ["AnimalId", "EventDate", "LactationNumber",
     "DaysInMilk", "ReproductionStatus"]
].dropna()

# Build a lookup: for each animal, get their records sorted by date
print("\nFilling identifier columns...")

filled_rows = []
for animal_id, group in df_missing[
    df_missing["AnimalId_Recovered"].notna()
].groupby("AnimalId_Recovered"):

    animal_known = known_ref[known_ref["AnimalId"] == animal_id].sort_values("EventDate")

    if animal_known.empty:
        continue

    for idx, row in group.iterrows():
        date_diffs = (animal_known["EventDate"] - row["EventDate"]).abs()
        closest    = animal_known.loc[date_diffs.idxmin()]
        filled_rows.append({
            "index":              idx,
            "LactationNumber":    closest["LactationNumber"],
            "DaysInMilk":         closest["DaysInMilk"],
            "ReproductionStatus": closest["ReproductionStatus"]
        })

df_filled = pd.DataFrame(filled_rows).set_index("index")
df_missing.loc[df_filled.index, "LactationNumber_Filled"]    = df_filled["LactationNumber"]
df_missing.loc[df_filled.index, "DaysInMilk_Filled"]         = df_filled["DaysInMilk"]
df_missing.loc[df_filled.index, "ReproductionStatus_Filled"] = df_filled["ReproductionStatus"]

print(f"LactationNumber filled    : {df_missing['LactationNumber_Filled'].notna().sum():,}")
print(f"DaysInMilk filled         : {df_missing['DaysInMilk_Filled'].notna().sum():,}")
print(f"ReproductionStatus filled : {df_missing['ReproductionStatus_Filled'].notna().sum():,}")

=== Recovery Summary ===


,count
RecoveryMethod,
Clustering,1646432
ExactMatch,47781
Unresolved,6790



Total recovered  : 1,694,213
Total unresolved : 6,790

Filling identifier columns...


KeyboardInterrupt: 